In [ ]:
import torch

# 为了方便起见，我们定义了一个计算卷积层的函数。
# 此函数初始化卷积层权重，并对输入和输出提高和缩减相应的维数
def comp_conv2d(conv2d, X):
    # 这里的（1，1）表示批量大小和通道数都是1
    X = X.reshape((1, 1) + X.shape) # 添加批量大小和通道维度,使输入成为四维张量
    Y = conv2d(X)
    # 省略前两个维度：批量大小和通道
    return Y.reshape(Y.shape[2:])

# 请注意，这里每边都填充了1行或1列，因此总共添加了2行或2列
conv2d = torch.nn.Conv2d(1, 1, kernel_size=3, padding=1, bias=False) # padding 是单侧填充。实际填充数为 padding * 2
X = torch.rand(size=(8, 8))

print(f'Input shape: {X.shape}')
print(f'Output shape: {comp_conv2d(conv2d, X).shape}')


Input shape: torch.Size([8, 8])
Output shape: torch.Size([8, 8])


当卷积核的高度和宽度不同时，我们可以填充不同的高度和宽度，使输出和输入具有相同的高度和宽度。在如下示例中，我们使用高度为5，宽度为3的卷积核，高度和宽度两边的填充分别为2和1。

In [4]:
conv2d = torch.nn.Conv2d(1, 1, kernel_size=(5, 3), padding=(2, 1))
comp_conv2d(conv2d, X).shape

torch.Size([8, 8])

In [6]:
conv2d = torch.nn.Conv2d(1, 1, kernel_size=3, padding=1, stride=2)
comp_conv2d(conv2d, X).shape

torch.Size([4, 4])

In [7]:
conv2d = torch.nn.Conv2d(1, 1, kernel_size=(3, 5), padding=(0, 1), stride=(3, 4))
comp_conv2d(conv2d, X).shape

torch.Size([2, 2])

6.4.1. 多输入通道

In [1]:
import torch

def corr2d(X, K):
    kh, kw = K.shape
    Y = torch.zeros((X.shape[0] - kh + 1, X.shape[1] - kw + 1))
    for i in range(Y.shape[0]):
        for j in range(Y.shape[1]):
            Y[i, j] = (X[i : i + kh, j : j + kw] * K).sum()
    return Y

def corr2d_multi_in(X, K):
    # 先遍历“X”和“K”的第0个维度（通道维度），再把它们加在一起
    return sum(corr2d(x, k) for x, k in zip(X, K)) # zip()函数将两个可迭代对象打包成一个元组的迭代器。然后，sum()函数将所有的二维卷积结果相加。

X = torch.tensor([[[0.0, 1.0, 2.0], [3.0, 4.0, 5.0], [6.0, 7.0, 8.0]],
               [[1.0, 2.0, 3.0], [4.0, 5.0, 6.0], [7.0, 8.0, 9.0]]])
K = torch.tensor([[[0.0, 1.0], [2.0, 3.0]], [[1.0, 2.0], [3.0, 4.0]]])

print(f'Input shape: {X.shape}')
print(f'Kernel shape: {K.shape}')
corr2d_multi_in(X, K)

Input shape: torch.Size([2, 3, 3])
Kernel shape: torch.Size([2, 2, 2])


tensor([[ 56.,  72.],
        [104., 120.]])

6.4.2. 多输出通道

In [2]:
def corr2d_multi_in_out(X, K):
    # 迭代“K”的第0个维度，每次都对输入“X”执行互相关运算。
    # 最后将所有结果都叠加在一起
    return torch.stack([corr2d_multi_in(X, k) for k in K], 0)

K = torch.stack((K, K + 1, K + 2), 0) # 此处的0表示在第0个维度上堆叠 

print(f'Kernel shape: {K.shape}')
print(f'X shape: {X.shape}')

corr2d_multi_in_out(X, K)
print(f'Output shape: {corr2d_multi_in_out(X, K).shape}')
print(f'Output: {corr2d_multi_in_out(X, K)}')

Kernel shape: torch.Size([3, 2, 2, 2])
X shape: torch.Size([2, 3, 3])
Output shape: torch.Size([3, 2, 2])
Output: tensor([[[ 56.,  72.],
         [104., 120.]],

        [[ 76., 100.],
         [148., 172.]],

        [[ 96., 128.],
         [192., 224.]]])
